# ACIF104 – Aprendizaje de máquina · Sumativa 1 (Semana 6) · Alineación de personajes DC

**Notebook único y reproducible.** Todas las tablas, cifras y figuras del informe salen de una sola ejecución completa de este cuaderno (*Entorno de ejecución → Ejecutar todo*). Los resultados se guardan en la carpeta `salidas/` y se descargan como `salidas_acif104.zip` al final.

Estructura:
1. Entorno y carga de datos
2. Calidad de datos (completitud, correctitud/consistencia, valores atípicos)
3. Variable objetivo y análisis exploratorio (patrones y relación con la alineación)
4. Preprocesamiento, partición 70/30 y verificación del balance de clases
5. Modelos de ML supervisado (línea base, regresión logística, árbol, random forest, SVM)
6. Técnicas no supervisadas (PCA y K-means)
7. Arquitecturas de DL (MLP denso, red con embeddings, ResNet tabular)
8. Comparación ML vs DL (validación cruzada) y refinamiento de la red seleccionada
9. Efecto del balanceo de clases (verificación + 4 técnicas, validación cruzada)
10. Evaluación final en el 30 % de prueba: tabla con todos los resultados y ganador
11. Importancia de variables por permutación
12. Explicabilidad con SHAP (global y casos particulares)
13. Exportación de resultados y del modelo para la aplicación

In [ ]:
# ============================================================
# 1. ENTORNO Y CARGA DE DATOS
# ============================================================
import os, sys, json, time, platform, warnings, zipfile
warnings.filterwarnings("ignore")

try:
    import imblearn, shap
except ImportError:
    !pip -q install imbalanced-learn shap
    import imblearn, shap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import sklearn
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             roc_curve, silhouette_score, adjusted_rand_score)
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import RandomOverSampler, SMOTENC
from imblearn.under_sampling import RandomUnderSampler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEMILLA = 42
def fijar_semillas(s=SEMILLA):
    np.random.seed(s)
    tf.keras.utils.set_random_seed(s)   # fija random, numpy y tensorflow
fijar_semillas()
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

OUT = "salidas"
os.makedirs(f"{OUT}/figuras", exist_ok=True)
os.makedirs(f"{OUT}/tablas", exist_ok=True)
RESUMEN = {}   # cifras clave que se copian al informe

def guardar_tabla(df, nombre, index=False):
    df.to_csv(f"{OUT}/tablas/{nombre}.csv", index=index, encoding="utf-8")

def guardar_fig(nombre):
    plt.tight_layout()
    plt.savefig(f"{OUT}/figuras/{nombre}.png", dpi=200, bbox_inches="tight")
    plt.show()

sns.set_theme(style="whitegrid", font_scale=0.95)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# --- Registro del hardware y versiones (va al informe) ---
gpus = tf.config.list_physical_devices("GPU")
entorno = {
    "python": platform.python_version(),
    "sistema": platform.platform(),
    "procesador": platform.processor() or platform.machine(),
    "nucleos_cpu": os.cpu_count(),
    "gpu": [g.name for g in gpus] if gpus else "sin GPU (CPU)",
    "pandas": pd.__version__, "numpy": np.__version__,
    "scikit_learn": sklearn.__version__, "imbalanced_learn": imblearn.__version__,
    "tensorflow": tf.__version__, "shap": shap.__version__,
}
try:
    import psutil
    entorno["ram_gb"] = round(psutil.virtual_memory().total / 1e9, 1)
except Exception:
    pass
RESUMEN["entorno"] = entorno
for k, v in entorno.items():
    print(f"{k:>18}: {v}")

# --- Carga del dataset ---
csv_path = "dc-wikia-data.csv"
if not os.path.exists(csv_path):
    csv_path = "https://raw.githubusercontent.com/fivethirtyeight/data/master/comic-characters/dc-wikia-data.csv"
df = pd.read_csv(csv_path)
print(f"\nDataset cargado desde: {csv_path}")
print("Dimensiones:", df.shape)
df.head()

## 2. Calidad de datos
Se revisa **completitud** (faltantes por columna), **correctitud/consistencia** (duplicados, categorías, rango del año y coherencia entre `YEAR` y `FIRST APPEARANCE`) y **valores atípicos** (regla de Tukey, 1,5·RIC) en `APPEARANCES` y `YEAR`.

In [ ]:
# ============================================================
# 2.1 Tabla de variables: tipo, únicos, % faltantes
# ============================================================
descripcion = {
    "page_id": "Identificador de la página wiki",
    "name": "Nombre del personaje",
    "urlslug": "Ruta URL de la página",
    "ID": "Tipo de identidad (secreta, pública, desconocida)",
    "ALIGN": "Alineación (bueno, malo, neutral, reformado)",
    "EYE": "Color de ojos",
    "HAIR": "Color de pelo",
    "SEX": "Sexo del personaje",
    "GSM": "Minoría sexual o de género",
    "ALIVE": "Vivo o fallecido",
    "APPEARANCES": "Número de apariciones en cómics",
    "FIRST APPEARANCE": "Mes y año de primera aparición (texto)",
    "YEAR": "Año de primera aparición",
}
rol = {"page_id": "Identificador (excluida)", "name": "Identificador (excluida)",
       "urlslug": "Identificador (excluida)", "ALIGN": "Objetivo",
       "FIRST APPEARANCE": "Redundante con YEAR (excluida)"}
filas = []
for c in df.columns:
    s = df[c]
    tipo = "Numérica" if pd.api.types.is_numeric_dtype(s) else "Categórica"
    if c in ("page_id",): tipo = "Numérica (ID)"
    filas.append({
        "Variable": c, "Descripción": descripcion.get(c, ""), "Tipo": tipo,
        "Valores únicos": s.nunique(dropna=True),
        "Faltantes (n)": int(s.isna().sum()),
        "Faltantes (%)": round(100 * s.isna().mean(), 2),
        "Rol": rol.get(c, "Predictora candidata"),
    })
tabla_vars = pd.DataFrame(filas)
guardar_tabla(tabla_vars, "t01_variables")
display(tabla_vars)

# Gráfico de completitud
plt.figure(figsize=(8, 4))
orden = tabla_vars.sort_values("Faltantes (%)", ascending=False)
sns.barplot(data=orden, y="Variable", x="Faltantes (%)", color="#2a7fb8")
for i, v in enumerate(orden["Faltantes (%)"]):
    plt.text(v + 0.5, i, f"{v:.1f} %", va="center", fontsize=8)
plt.title("Porcentaje de valores faltantes por variable (dataset completo)")
plt.xlabel("% faltante"); plt.ylabel("")
guardar_fig("f01_faltantes")

In [ ]:
# ============================================================
# 2.2 Correctitud y consistencia
# ============================================================
calidad = {}
calidad["filas"] = int(len(df))
calidad["filas_duplicadas"] = int(df.duplicated().sum())
calidad["page_id_duplicados"] = int(df["page_id"].duplicated().sum())
calidad["name_duplicados"] = int(df["name"].duplicated().sum())

# Rango del año: los cómics de DC comienzan en 1935 y el dataset se compiló en 2014
calidad["year_min"] = float(df["YEAR"].min()); calidad["year_max"] = float(df["YEAR"].max())
calidad["year_fuera_de_rango"] = int(((df["YEAR"] < 1935) | (df["YEAR"] > 2014)).sum())
# Apariciones imposibles (<= 0) o no enteras
ap = df["APPEARANCES"]
calidad["apariciones_min"] = float(ap.min()); calidad["apariciones_max"] = float(ap.max())
calidad["apariciones_no_positivas"] = int((ap <= 0).sum())
calidad["apariciones_no_enteras"] = int((ap.dropna() % 1 != 0).sum())

# Coherencia YEAR vs año contenido en FIRST APPEARANCE
anio_fa = pd.to_numeric(df["FIRST APPEARANCE"].astype(str).str.extract(r"(\d{4})")[0], errors="coerce")
ambos = df["YEAR"].notna() & anio_fa.notna()
calidad["year_vs_first_appearance_comparables"] = int(ambos.sum())
calidad["year_vs_first_appearance_inconsistentes"] = int((df.loc[ambos, "YEAR"] != anio_fa[ambos]).sum())

for k, v in calidad.items():
    print(f"{k:>42}: {v}")
RESUMEN["calidad"] = calidad
guardar_tabla(pd.Series(calidad, name="valor").rename_axis("indicador").reset_index(), "t02_calidad")

# Categorías presentes (para revisar etiquetas inconsistentes)
for c in ["ALIGN", "ID", "SEX", "ALIVE", "GSM"]:
    print(f"\n{c}:\n", df[c].value_counts(dropna=False).to_string())
print("\nEYE (primeras 12):\n", df["EYE"].value_counts(dropna=False).head(12).to_string())
print("\nHAIR (primeras 12):\n", df["HAIR"].value_counts(dropna=False).head(12).to_string())

In [ ]:
# ============================================================
# 2.3 Estadística descriptiva y valores atípicos (Tukey 1,5·RIC)
# ============================================================
num = ["APPEARANCES", "YEAR"]
desc = df[num].describe(percentiles=[.25, .5, .75, .95, .99]).T
desc["asimetría"] = df[num].skew()
desc["curtosis"] = df[num].kurt()
guardar_tabla(desc.round(2), "t03_descriptivos_numericas", index=True)
display(desc.round(2))

filas = []
for c in num:
    s = df[c].dropna()
    q1, q3 = s.quantile([.25, .75]); ric = q3 - q1
    li, ls = q1 - 1.5 * ric, q3 + 1.5 * ric
    n_out = int(((s < li) | (s > ls)).sum())
    filas.append({"Variable": c, "Q1": q1, "Q3": q3, "RIC": ric,
                  "Límite inferior": li, "Límite superior": ls,
                  "Atípicos (n)": n_out, "Atípicos (%)": round(100 * n_out / len(s), 2)})
tabla_out = pd.DataFrame(filas)
guardar_tabla(tabla_out.round(2), "t04_atipicos")
display(tabla_out.round(2))
RESUMEN["atipicos"] = tabla_out.round(2).to_dict(orient="records")

top_ap = df.nlargest(5, "APPEARANCES")[["name", "ALIGN", "APPEARANCES", "YEAR"]]
guardar_tabla(top_ap, "t05_top_apariciones")
display(top_ap)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
sns.boxplot(x=df["APPEARANCES"], ax=ax[0], color="#2a7fb8")
ax[0].set_title("Apariciones (escala original)")
sns.boxplot(x=np.log10(df["APPEARANCES"]), ax=ax[1], color="#2a7fb8")
ax[1].set_title("Apariciones (log10)"); ax[1].set_xlabel("log10(APPEARANCES)")
sns.boxplot(x=df["YEAR"], ax=ax[2], color="#e08a2c")
ax[2].set_title("Año de primera aparición")
guardar_fig("f02_atipicos_boxplot")

## 3. Variable objetivo y análisis exploratorio
El problema se plantea como **clasificación binaria**: personajes buenos (1) frente a malos (0). Se descartan neutrales, reformados y registros sin alineación.

In [ ]:
# ============================================================
# 3.1 Definición del objetivo y distribución de clases
# ============================================================
datos = df[df["ALIGN"].isin(["Good Characters", "Bad Characters"])].copy()
datos["target"] = (datos["ALIGN"] == "Good Characters").astype(int)
dist = datos["target"].value_counts().sort_index()
tabla_clases = pd.DataFrame({"Clase": ["Malo (0)", "Bueno (1)"],
                             "Registros": dist.values,
                             "Porcentaje": (100 * dist / dist.sum()).round(2).values})
guardar_tabla(tabla_clases, "t06_clases")
display(tabla_clases)
RESUMEN["objetivo"] = {"registros_binarios": int(len(datos)),
                       "excluidos": int(len(df) - len(datos)),
                       "malos": int(dist[0]), "buenos": int(dist[1]),
                       "ratio_desbalance": round(dist.max() / dist.min(), 3)}
print(RESUMEN["objetivo"])

# Faltantes dentro del subconjunto binario (lo que efectivamente se modela)
falt_bin = (100 * datos.drop(columns=["target"]).isna().mean()).round(2).rename("Faltantes (%) subconjunto binario")
guardar_tabla(falt_bin.rename_axis("Variable").reset_index(), "t07_faltantes_binario")
display(falt_bin.to_frame())

In [ ]:
# ============================================================
# 3.2 Variables numéricas frente a la alineación
# ============================================================
etq = {0: "Malo", 1: "Bueno"}
datos["Alineación"] = datos["target"].map(etq)
datos["log10_APARICIONES"] = np.log10(datos["APPEARANCES"])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=datos, x="log10_APARICIONES", hue="Alineación", bins=30, stat="density",
             common_norm=False, element="step", ax=ax[0], palette={"Malo": "#c0392b", "Bueno": "#2a7fb8"})
ax[0].set_title("Distribución de apariciones (log10) por alineación")
sns.histplot(data=datos, x="YEAR", hue="Alineación", bins=40, stat="density",
             common_norm=False, element="step", ax=ax[1], palette={"Malo": "#c0392b", "Bueno": "#2a7fb8"})
ax[1].set_title("Distribución del año de primera aparición por alineación")
guardar_fig("f03_numericas_por_clase")

# Proporción de buenos por década (tendencia temporal)
datos["Década"] = (datos["YEAR"] // 10 * 10)
por_decada = datos.groupby("Década")["target"].agg(["mean", "size"]).reset_index()
fig, ax1 = plt.subplots(figsize=(9, 3.8))
ax1.bar(por_decada["Década"], por_decada["size"], width=7, color="#cfd8e3", label="Personajes")
ax1.set_ylabel("Personajes"); ax1.set_xlabel("Década de primera aparición")
ax2 = ax1.twinx()
ax2.plot(por_decada["Década"], 100 * por_decada["mean"], marker="o", color="#2a7fb8", label="% buenos")
ax2.axhline(50, ls="--", color="grey", lw=0.8); ax2.set_ylabel("% buenos"); ax2.set_ylim(0, 100)
ax2.grid(False)
plt.title("Personajes y porcentaje de buenos por década")
guardar_fig("f04_tendencia_decada")
guardar_tabla(por_decada.rename(columns={"mean": "prop_buenos", "size": "n"}), "t08_por_decada")

# Pruebas de Mann-Whitney (no paramétrica: apariciones muy asimétricas)
filas = []
for c in ["APPEARANCES", "YEAR"]:
    a = datos.loc[datos.target == 1, c].dropna(); b = datos.loc[datos.target == 0, c].dropna()
    u, p = stats.mannwhitneyu(a, b)
    filas.append({"Variable": c, "Mediana buenos": a.median(), "Mediana malos": b.median(),
                  "U": u, "p-valor": p, "Correlación biserial de rangos": 1 - 2 * u / (len(a) * len(b))})
tabla_mw = pd.DataFrame(filas)
guardar_tabla(tabla_mw, "t09_mannwhitney")
display(tabla_mw)

In [ ]:
# ============================================================
# 3.3 Variables categóricas frente a la alineación
# ============================================================
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(ct, correction=False)[0]
    n = ct.values.sum(); r, k = ct.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1))) if min(r, k) > 1 else np.nan

cats = ["SEX", "ALIVE", "ID", "EYE", "HAIR", "GSM"]
tmp = datos.copy()
for c in cats:
    tmp[c] = tmp[c].fillna("Faltante")

filas = []
for c in cats:
    ct = pd.crosstab(tmp[c], tmp["target"])
    chi2, p, gl, _ = stats.chi2_contingency(ct)
    filas.append({"Variable": c, "Categorías": ct.shape[0], "Chi²": chi2, "gl": gl,
                  "p-valor": p, "V de Cramér": cramers_v(tmp[c], tmp["target"])})
tabla_chi = pd.DataFrame(filas).sort_values("V de Cramér", ascending=False)
guardar_tabla(tabla_chi, "t10_chi2_cramer")
display(tabla_chi)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, c in zip(axes.ravel(), cats):
    top = tmp[c].value_counts().head(8).index
    g = tmp[tmp[c].isin(top)].groupby(c)["target"].agg(["mean", "size"]).sort_values("mean")
    ax.barh(g.index.astype(str), 100 * g["mean"], color="#2a7fb8")
    for i, (m, n) in enumerate(zip(g["mean"], g["size"])):
        ax.text(100 * m + 1, i, f"n={n}", va="center", fontsize=7)
    ax.axvline(100 * datos["target"].mean(), ls="--", color="grey", lw=0.8)
    ax.set_xlim(0, 110); ax.set_title(f"% de buenos según {c}"); ax.set_xlabel("% buenos")
guardar_fig("f05_categoricas_por_clase")

# Mapa de calor de asociación (V de Cramér) entre todas las variables y el objetivo
tmp["APARICIONES_cuartil"] = pd.qcut(tmp["APPEARANCES"], 4, labels=False, duplicates="drop").astype("Int64").astype(str)
tmp["DÉCADA"] = tmp["Década"].astype("Int64").astype(str)
vars_asoc = cats + ["APARICIONES_cuartil", "DÉCADA", "target"]
M = pd.DataFrame(index=vars_asoc, columns=vars_asoc, dtype=float)
for a in vars_asoc:
    for b in vars_asoc:
        M.loc[a, b] = 1.0 if a == b else cramers_v(tmp[a], tmp[b])
plt.figure(figsize=(8, 6.5))
sns.heatmap(M.astype(float), annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, cbar_kws={"label": "V de Cramér"})
plt.title("Asociación entre variables (V de Cramér)")
guardar_fig("f06_mapa_calor_asociacion")
guardar_tabla(M.round(3), "t11_matriz_cramer", index=True)

## 4. Preprocesamiento y partición 70/30
Decisiones derivadas del EDA:
- Se excluyen identificadores (`page_id`, `name`, `urlslug`) y `FIRST APPEARANCE` (redundante con `YEAR`).
- Se excluyen variables con más de 90 % de faltantes en el subconjunto modelado (se verifica en código).
- `APPEARANCES` se transforma con log(1+x) por su fuerte asimetría; los atípicos **no se eliminan** porque son valores reales (personajes muy populares), no errores de registro.
- Numéricas: imputación por mediana + indicador de faltante + estandarización.
- Categóricas: categoría explícita `Faltante` + **one-hot** (categorías con menos de 10 casos se agrupan como infrecuentes).

**Partición:** 70 % entrenamiento y 30 % prueba, estratificada. La selección de modelos e hiperparámetros se hace **solo dentro del 70 %** mediante validación cruzada estratificada de 5 pliegues; en cada pliegue el preprocesamiento y el balanceo se ajustan únicamente con la parte de entrenamiento del pliegue. El 30 % de prueba se usa una sola vez, al final, para verificar que el modelo aprendió.

In [ ]:
# ============================================================
# 4. PREPROCESAMIENTO Y PARTICIÓN 70/30
# ============================================================
from sklearn.model_selection import StratifiedKFold

cand_cat = ["ID", "EYE", "HAIR", "SEX", "ALIVE", "GSM"]
cand_num = ["APPEARANCES", "YEAR"]
falt = datos[cand_cat + cand_num].isna().mean()
excluidas_faltantes = [c for c in cand_cat + cand_num if falt[c] > 0.90]
CAT = [c for c in cand_cat if c not in excluidas_faltantes]
NUM = [c for c in cand_num if c not in excluidas_faltantes]
print("Excluidas por >90 % faltantes:", excluidas_faltantes)
print("Categóricas:", CAT, "| Numéricas:", NUM)

X = datos[CAT + NUM].copy()
y = datos["target"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEMILLA)

tabla_part = pd.DataFrame({
    "Conjunto": ["Entrenamiento", "Prueba"],
    "Registros": [len(y_train), len(y_test)],
    "Proporción (%)": [round(100 * len(v) / len(y), 1) for v in (y_train, y_test)],
    "% buenos": [round(100 * v.mean(), 2) for v in (y_train, y_test)],
    "% malos": [round(100 * (1 - v.mean()), 2) for v in (y_train, y_test)],
})
guardar_tabla(tabla_part, "t12_particion")
display(tabla_part)

# ¿Están balanceadas las clases? (criterio: la minoritaria representa al menos 40 %)
prop_min = min(y_train.mean(), 1 - y_train.mean())
BALANCEADAS = prop_min >= 0.40
print(f"Clase minoritaria en entrenamiento: {100*prop_min:.2f} % -> "
      f"{'clases balanceadas' if BALANCEADAS else 'clases desbalanceadas: se requiere técnica de balanceo'}")

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
PLIEGUES = list(CV.split(X_train, y_train))

def transformadores_numericos():
    """APPEARANCES: log(1+x) por su asimetría; YEAR: sin transformar. Ambas: mediana + indicador de faltante + escalado.
    Se usa np.log1p (no una función propia) para que el preprocesador guardado se pueda cargar en la aplicación."""
    t = []
    if "APPEARANCES" in NUM:
        t.append(("apar", Pipeline([
            ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
            ("imp", SimpleImputer(strategy="median", add_indicator=True)),
            ("esc", StandardScaler())]), ["APPEARANCES"]))
    otras = [c for c in NUM if c != "APPEARANCES"]
    if otras:
        t.append(("num", Pipeline([
            ("imp", SimpleImputer(strategy="median", add_indicator=True)),
            ("esc", StandardScaler())]), otras))
    return t

def crear_preprocesador():
    cat_pipe = Pipeline([
        ("imp", SimpleImputer(strategy="constant", fill_value="Faltante")),
        ("ohe", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=10, sparse_output=False)),
    ])
    return ColumnTransformer(transformadores_numericos() + [("cat", cat_pipe, CAT)])

prep_total = crear_preprocesador().fit(X_train)
print(f"Dimensión tras one-hot (ajustado en el 70 %): {prep_total.transform(X_train).shape[1]} columnas")
RESUMEN["preprocesamiento"] = {"categoricas": CAT, "numericas": NUM,
                                "excluidas_por_faltantes": excluidas_faltantes,
                                "n_features_onehot": int(prep_total.transform(X_train).shape[1]),
                                "particion": tabla_part.to_dict(orient="records"),
                                "prop_minoritaria_train": round(float(prop_min), 4),
                                "clases_balanceadas": bool(BALANCEADAS)}

In [ ]:
# ============================================================
# 4b. FUNCIONES COMUNES: métricas, balanceo y validación cruzada
# ============================================================
def metricas(y_true, y_pred, score=None):
    m = {"Exactitud": accuracy_score(y_true, y_pred),
         "Precisión": precision_score(y_true, y_pred, zero_division=0),
         "Recall": recall_score(y_true, y_pred, zero_division=0),
         "F1": f1_score(y_true, y_pred, zero_division=0)}
    m["ROC-AUC"] = roc_auc_score(y_true, score) if score is not None else np.nan
    return m

METRICAS = ["Exactitud", "Precisión", "Recall", "F1", "ROC-AUC"]

def remuestrear(estrategia, Xdf, yv, semilla=SEMILLA):
    """Aplica la estrategia SOLO a los datos de entrenamiento recibidos. Devuelve (X, y, class_weight)."""
    yv = np.asarray(yv)
    if estrategia == "Sin balanceo":
        return Xdf, yv, None
    if estrategia == "Pesos de clase":
        w = compute_class_weight("balanced", classes=np.array([0, 1]), y=yv)
        return Xdf, yv, {0: w[0], 1: w[1]}
    if estrategia == "Sobremuestreo aleatorio":      # duplica registros de la clase minoritaria
        Xr, yr = RandomOverSampler(random_state=semilla).fit_resample(Xdf, yv)
        return Xr, np.asarray(yr), None
    if estrategia == "Submuestreo aleatorio":        # elimina registros de la clase mayoritaria
        Xr, yr = RandomUnderSampler(random_state=semilla).fit_resample(Xdf, yv)
        return Xr, np.asarray(yr), None
    if estrategia == "SMOTE-NC":                     # genera registros sintéticos (variables mixtas)
        ic = SimpleImputer(strategy="constant", fill_value="Faltante").fit(Xdf[CAT])
        im = SimpleImputer(strategy="median").fit(Xdf[NUM])
        Xi = pd.concat([pd.DataFrame(ic.transform(Xdf[CAT]), columns=CAT),
                        pd.DataFrame(im.transform(Xdf[NUM]), columns=NUM)], axis=1)
        Xr, yr = SMOTENC(categorical_features=list(range(len(CAT))), random_state=semilla).fit_resample(Xi, yv)
        Xr = pd.DataFrame(Xr, columns=CAT + NUM)
        for c in NUM: Xr[c] = Xr[c].astype(float)
        return Xr, np.asarray(yr), None
    raise ValueError(estrategia)

def validar_cruzado(ajustar, predecir, estrategia="Sin balanceo"):
    """Validación cruzada estratificada de 5 pliegues dentro del 70 %.
    ajustar(Xtr_df, ytr, Xva_df, yva, cw, semilla) -> objeto; predecir(obj, Xdf) -> (pred, score)."""
    filas, extras = [], []
    for k, (itr, iva) in enumerate(PLIEGUES):
        Xa, ya = X_train.iloc[itr], y_train[itr]
        Xv, yv = X_train.iloc[iva], y_train[iva]
        Xr, yr, cw = remuestrear(estrategia, Xa, ya, semilla=k)
        obj = ajustar(Xr, yr, Xv, yv, cw, k)
        pred, sc = predecir(obj, Xv)
        filas.append({"Pliegue": k + 1, **metricas(yv, pred, sc)})
        extras.append(obj)
    return pd.DataFrame(filas), extras

def resumir_cv(df_cv):
    r = {}
    for m in METRICAS:
        r[f"{m} media"] = df_cv[m].mean(); r[f"{m} DE"] = df_cv[m].std()
    return r

## 5. Modelos de ML supervisado
Se compara una **línea base** (clase mayoritaria) con regresión logística, árbol de decisión, random forest y SVM con kernel RBF. Los hiperparámetros se eligen con una rejilla pequeña según el **F1 medio de la validación cruzada** (5 pliegues, solo con el 70 %).

In [ ]:
# ============================================================
# 5. MODELOS DE ML (selección por validación cruzada)
# ============================================================
def puntaje(modelo, Xm):
    if hasattr(modelo, "predict_proba"):
        return modelo.predict_proba(Xm)[:, 1]
    return modelo.decision_function(Xm)

def fabrica_ml(Clase, params):
    def ajustar(Xr, yr, Xv, yv, cw, semilla):
        p = dict(params)
        if cw is not None: p["class_weight"] = cw
        prep = crear_preprocesador().fit(Xr)
        return prep, Clase(**p).fit(prep.transform(Xr), yr)
    def predecir(obj, Xdf):
        prep, mod = obj
        Xm = prep.transform(Xdf)
        sc = None if isinstance(mod, DummyClassifier) else puntaje(mod, Xm)
        return mod.predict(Xm), sc
    return ajustar, predecir

modelos_ml = {
    "Línea base (mayoritaria)": (DummyClassifier, {"strategy": ["most_frequent"]}),
    "Regresión logística": (LogisticRegression, {"C": [0.01, 0.1, 1, 10], "max_iter": [2000]}),
    "Árbol de decisión": (DecisionTreeClassifier, {"max_depth": [3, 5, 8, 12], "min_samples_leaf": [5, 20], "random_state": [SEMILLA]}),
    "Random forest": (RandomForestClassifier, {"n_estimators": [300], "max_depth": [None, 10, 20], "min_samples_leaf": [1, 5], "random_state": [SEMILLA], "n_jobs": [-1]}),
    "SVM (RBF)": (SVC, {"C": [0.1, 1, 10], "gamma": ["scale"], "kernel": ["rbf"], "random_state": [SEMILLA]}),
}
ocultar = ("random_state", "n_jobs", "max_iter", "kernel", "strategy")

res_ml, mejores_ml, grid_log, cv_ml = [], {}, [], {}
for nombre, (Clase, grid) in modelos_ml.items():
    mejor = None
    for params in ParameterGrid(grid):
        t0 = time.time()
        df_cv, _ = validar_cruzado(*fabrica_ml(Clase, params))
        t = (time.time() - t0) / len(PLIEGUES)
        r = resumir_cv(df_cv)
        etiqueta = str({k: v for k, v in params.items() if k not in ocultar})
        grid_log.append({"Modelo": nombre, "Hiperparámetros": etiqueta, **r})
        if mejor is None or r["F1 media"] > mejor[1]["F1 media"]:
            mejor = (params, r, df_cv, t, etiqueta)
    params, r, df_cv, t, etiqueta = mejor
    mejores_ml[nombre] = (Clase, params)
    cv_ml[nombre] = df_cv
    res_ml.append({"Familia": "ML", "Modelo": nombre, "Hiperparámetros": etiqueta, **r,
                   "Tiempo por pliegue (s)": round(t, 2)})
    print(f"{nombre}: F1 CV = {r['F1 media']:.4f} ± {r['F1 DE']:.4f}  {etiqueta}")

tabla_ml = pd.DataFrame(res_ml)
guardar_tabla(tabla_ml.round(4), "t13_ml_cv")
guardar_tabla(pd.DataFrame(grid_log).round(4), "t14_ml_rejilla_cv")
display(tabla_ml.round(4))

## 6. Técnicas no supervisadas
PCA para visualizar la separabilidad de las clases y K-means para comprobar si los grupos naturales de los datos coinciden con la alineación (solo con el 70 % de entrenamiento). Esto indica si un enfoque no supervisado sirve para el objetivo.

In [ ]:
# ============================================================
# 6. NO SUPERVISADO: PCA + K-MEANS
# ============================================================
etq = {0: "Malo", 1: "Bueno"}
Xtr_all = prep_total.transform(X_train)
pca = PCA(n_components=2, random_state=SEMILLA).fit(Xtr_all)
Z = pca.transform(Xtr_all)
var_exp = pca.explained_variance_ratio_
plt.figure(figsize=(7, 5))
for k, col in [(0, "#c0392b"), (1, "#2a7fb8")]:
    plt.scatter(Z[y_train == k, 0], Z[y_train == k, 1], s=6, alpha=0.4, c=col, label=etq[k])
plt.xlabel(f"CP1 ({100*var_exp[0]:.1f} % var.)"); plt.ylabel(f"CP2 ({100*var_exp[1]:.1f} % var.)")
plt.title("Proyección PCA del conjunto de entrenamiento"); plt.legend(markerscale=3)
guardar_fig("f07_pca")

filas = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(Xtr_all)
    filas.append({"k": k, "Silueta": silhouette_score(Xtr_all, km.labels_, sample_size=3000, random_state=SEMILLA),
                  "ARI vs alineación": adjusted_rand_score(y_train, km.labels_)})
tabla_km = pd.DataFrame(filas)
guardar_tabla(tabla_km.round(4), "t15_kmeans")
display(tabla_km.round(4))
RESUMEN["no_supervisado"] = {"pca_var_explicada_2cp": round(float(var_exp.sum()), 4),
                             "kmeans": tabla_km.round(4).to_dict(orient="records")}

## 7. Arquitecturas de Deep Learning
Tres arquitecturas **distintas** (no solo variaciones de profundidad):
- **A. MLP denso** sobre la entrada one-hot (128-64-32, ReLU, Dropout).
- **B. Red con embeddings de entidad**: cada variable categórica entra por su propia capa `Embedding` (Guo y Berkhahn, 2016) y se concatena con las numéricas.
- **C. ResNet tabular**: bloques residuales con normalización por lotes y conexiones de salto (Gorishniy et al., 2021).

**Justificación del diseño (común a las tres):**
- *Entrada:* variables preprocesadas (one-hot + numéricas estandarizadas); en B, códigos enteros por variable categórica más las numéricas.
- *Salida:* una neurona sigmoide, porque el objetivo es binario y entrega la probabilidad de ser "bueno"; pérdida de entropía cruzada binaria.
- *Capas ocultas ReLU* en embudo (128 → 64 → 32): con unos 4.000 registros de entrenamiento y pocas decenas de entradas, una red más grande sobreajusta; el embudo comprime gradualmente la representación.
- *Dropout* como regularización y *lotes de 128*: unas 25 actualizaciones por época, equilibrio entre estabilidad del gradiente y velocidad. Lote, tasa y Dropout se revisan en el refinamiento (sección 8b).

**Entrenamiento:** Adam, máximo 100 épocas y detención temprana (paciencia 10) sobre la parte de validación de cada pliegue, con restauración de los mejores pesos. El modelo final se entrena con todo el 70 % durante la **mediana de las mejores épocas** observadas en la validación cruzada (así no se usa la prueba para detenerlo).

In [ ]:
# ============================================================
# 7. DEEP LEARNING
# ============================================================
def construir(arquitectura, n_in, cardinalidades=None, dropout=0.2):
    if arquitectura == "A. MLP denso":
        inp = keras.Input(shape=(n_in,)); x = inp
        for u in (128, 64, 32):
            x = layers.Dense(u, activation="relu")(x); x = layers.Dropout(dropout)(x)
        return keras.Model(inp, layers.Dense(1, activation="sigmoid")(x))
    if arquitectura == "B. Red con embeddings":
        ins, embs = [], []
        for nombre, card in zip(CAT, cardinalidades):
            i = keras.Input(shape=(1,), dtype="int32", name=f"in_{nombre}")
            d = int(min(10, (card + 1) // 2))
            embs.append(layers.Flatten()(layers.Embedding(card, d, name=f"emb_{nombre}")(i))); ins.append(i)
        i_num = keras.Input(shape=(n_in,), name="in_num"); ins.append(i_num)
        x = layers.Concatenate()(embs + [i_num])
        for u in (64, 32):
            x = layers.Dense(u, activation="relu")(x); x = layers.Dropout(dropout)(x)
        return keras.Model(ins, layers.Dense(1, activation="sigmoid")(x))
    if arquitectura == "C. ResNet tabular":
        inp = keras.Input(shape=(n_in,))
        x = layers.Dense(64)(inp)
        for _ in range(2):
            h = layers.BatchNormalization()(x)
            h = layers.Dense(128, activation="relu")(h); h = layers.Dropout(dropout)(h)
            h = layers.Dense(64)(h)
            x = layers.Add()([x, h])
        x = layers.Activation("relu")(layers.BatchNormalization()(x))
        return keras.Model(inp, layers.Dense(1, activation="sigmoid")(x))
    raise ValueError(arquitectura)

class Codificador:
    """Ajusta la codificación de entrada SOLO con los datos de entrenamiento recibidos."""
    def __init__(self, arquitectura, Xdf):
        self.emb = arquitectura.startswith("B")
        if self.emb:
            self.ord = Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Faltante")),
                                 ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, min_frequency=10))]).fit(Xdf[CAT])
            self.num = ColumnTransformer(transformadores_numericos()).fit(Xdf)
            self.card = [len(c) + 1 for c in self.ord.named_steps["ord"].categories_]
        else:
            self.prep = crear_preprocesador().fit(Xdf); self.card = None
    def __call__(self, Xdf):
        if self.emb:
            cod = self.ord.transform(Xdf[CAT]).astype("int32") + 1          # -1 (desconocida) -> 0
            return [cod[:, i] for i in range(len(CAT))] + [self.num.transform(Xdf).astype("float32")]
        return self.prep.transform(Xdf).astype("float32")
    def n_in(self, Xin):
        return Xin[-1].shape[1] if isinstance(Xin, list) else Xin.shape[1]

def entrenar_dl(arquitectura, Xr, yr, Xv=None, yv=None, cw=None, semilla=SEMILLA,
                lr=1e-3, batch=128, dropout=0.2, epocas=None):
    fijar_semillas(semilla)
    cod = Codificador(arquitectura, Xr)
    Xin = cod(Xr)
    modelo = construir(arquitectura, cod.n_in(Xin), cod.card, dropout)
    modelo.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy",
                   metrics=["accuracy", keras.metrics.AUC(name="auc")])
    if Xv is not None:   # dentro de la validación cruzada: detención temprana con la parte de validación del pliegue
        es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
        h = modelo.fit(Xin, np.asarray(yr), validation_data=(cod(Xv), np.asarray(yv)), epochs=100,
                       batch_size=batch, callbacks=[es], class_weight=cw, verbose=0)
        mejor = int(np.argmin(h.history["val_loss"])) + 1
    else:                # modelo final: épocas fijadas por la validación cruzada
        h = modelo.fit(Xin, np.asarray(yr), epochs=epocas, batch_size=batch, class_weight=cw, verbose=0)
        mejor = epocas
    return {"cod": cod, "modelo": modelo, "hist": h.history, "mejor_epoca": mejor, "epocas": len(h.history["loss"])}

def predecir_dl(obj, Xdf):
    p = obj["modelo"].predict(obj["cod"](Xdf), verbose=0).ravel()
    return (p >= 0.5).astype(int), p

def fabrica_dl(arquitectura, **hp):
    def ajustar(Xr, yr, Xv, yv, cw, semilla):
        return entrenar_dl(arquitectura, Xr, yr, Xv, yv, cw=cw, semilla=semilla, **hp)
    return ajustar, predecir_dl

ARQUITECTURAS = ["A. MLP denso", "B. Red con embeddings", "C. ResNet tabular"]
res_dl, cv_dl, epocas_dl, historias = [], {}, {}, {}
for arq in ARQUITECTURAS:
    t0 = time.time()
    df_cv, objs = validar_cruzado(*fabrica_dl(arq))
    t = (time.time() - t0) / len(PLIEGUES)
    cv_dl[arq] = df_cv
    epocas_dl[arq] = int(np.median([o["mejor_epoca"] for o in objs]))
    historias[arq] = [o["hist"] for o in objs]
    r = resumir_cv(df_cv)
    res_dl.append({"Familia": "DL", "Modelo": arq,
                   "Hiperparámetros": f"{objs[0]['modelo'].count_params()} parámetros; drop=0.2, lr=0.001, lote=128",
                   **r, "Tiempo por pliegue (s)": round(t, 2),
                   "Mejores épocas por pliegue": str([o["mejor_epoca"] for o in objs]),
                   "Épocas modelo final (mediana)": epocas_dl[arq]})
    print(f"{arq}: F1 CV = {r['F1 media']:.4f} ± {r['F1 DE']:.4f}")
tabla_dl = pd.DataFrame(res_dl)
guardar_tabla(tabla_dl.round(4), "t16_dl_cv")
display(tabla_dl.round(4))

# Curvas de convergencia (pliegue 1 de cada arquitectura)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, arq in zip(axes, ARQUITECTURAS):
    hh = historias[arq][0]
    ax.plot(hh["loss"], label="Entrenamiento"); ax.plot(hh["val_loss"], label="Validación")
    me = int(np.argmin(hh["val_loss"]))
    ax.axvline(me, ls="--", color="grey", lw=0.8)
    ax.set_title(f"{arq} (pliegue 1, mejor época {me+1})"); ax.set_xlabel("Época"); ax.set_ylabel("Pérdida")
    ax.legend()
guardar_fig("f08_curvas_dl")

## 8. Comparación ML vs DL (validación cruzada)

In [ ]:
# ============================================================
# 8. COMPARACIÓN EN VALIDACIÓN CRUZADA
# ============================================================
cols = ["Familia", "Modelo"] + [f"{m} {s}" for m in METRICAS for s in ("media", "DE")] + ["Tiempo por pliegue (s)"]
tabla_comp = pd.concat([tabla_ml, tabla_dl], ignore_index=True)[cols]
guardar_tabla(tabla_comp.round(4), "t17_comparacion_cv")
display(tabla_comp.round(4))

d = pd.concat([cv_ml[n].assign(Modelo=n) for n in cv_ml if not n.startswith("Línea")] +
              [cv_dl[a].assign(Modelo=a) for a in ARQUITECTURAS])
plt.figure(figsize=(11, 4.2))
sns.boxplot(data=d, x="Modelo", y="F1", color="#cfe3f3")
sns.stripplot(data=d, x="Modelo", y="F1", color="#2a7fb8", size=4)
base_f1 = tabla_ml.loc[tabla_ml.Modelo.str.startswith("Línea"), "F1 media"].iloc[0]
plt.xticks(rotation=20, ha="right"); plt.ylabel("F1 por pliegue (validación cruzada)"); plt.xlabel("")
plt.title("Comparación de modelos de ML y DL (5 pliegues)")
guardar_fig("f09_comparacion_modelos")

candidatos_ml = tabla_ml[~tabla_ml["Modelo"].str.startswith("Línea")]
MEJOR_ML = candidatos_ml.sort_values(["F1 media", "ROC-AUC media"], ascending=False).iloc[0]["Modelo"]
MEJOR_DL = tabla_dl.sort_values(["F1 media", "ROC-AUC media"], ascending=False).iloc[0]["Modelo"]
print("Mejor ML (F1 CV):", MEJOR_ML, "| Mejor DL (F1 CV):", MEJOR_DL)
RESUMEN["comparacion_cv"] = tabla_comp.round(4).to_dict(orient="records")
RESUMEN["mejor_ml"] = MEJOR_ML; RESUMEN["mejor_dl"] = MEJOR_DL

## 8b. Refinamiento de la arquitectura DL seleccionada
Se ajustan tres hiperparámetros de la mejor arquitectura DL: Dropout (0,1 y 0,3), tasa de aprendizaje (0,001 y 0,0003) y tamaño de lote (64 y 128), con la misma validación cruzada. Se elige la combinación con mayor F1 medio; esa configuración se usa en el balanceo y en el modelo final.

In [ ]:
# ============================================================
# 8b. REFINAMIENTO DE LA ARQUITECTURA DL SELECCIONADA
# ============================================================
filas, hist_ref, epocas_ref = [], {}, {}
for hp in ParameterGrid({"dropout": [0.1, 0.3], "lr": [1e-3, 3e-4], "batch": [64, 128]}):
    df_cv, objs = validar_cruzado(*fabrica_dl(MEJOR_DL, **hp))
    et = f"drop={hp['dropout']}, lr={hp['lr']}, lote={hp['batch']}"
    hist_ref[et] = objs[0]["hist"]
    epocas_ref[et] = int(np.median([o["mejor_epoca"] for o in objs]))
    filas.append({**hp, "Configuración": et, **resumir_cv(df_cv),
                  "Épocas (mediana)": epocas_ref[et]})
    print(et, f"F1 CV = {filas[-1]['F1 media']:.4f}")
tabla_ref = pd.DataFrame(filas).sort_values(["F1 media", "ROC-AUC media"], ascending=False)
vis = tabla_ref.drop(columns=["dropout", "lr", "batch"])
guardar_tabla(vis.round(4), "t16b_refinamiento_dl")
display(vis.round(4))
mejor = tabla_ref.iloc[0]
CONFIG_DL = {"dropout": float(mejor["dropout"]), "lr": float(mejor["lr"]), "batch": int(mejor["batch"])}
EPOCAS_DL_REF = int(mejor["Épocas (mediana)"])
print("Configuración DL refinada:", CONFIG_DL, "| épocas modelo final:", EPOCAS_DL_REF)
RESUMEN["refinamiento_dl"] = {"config": CONFIG_DL, "epocas_final": EPOCAS_DL_REF, "tabla": vis.round(4).to_dict(orient="records")}

plt.figure(figsize=(9, 4))
for et, hh in hist_ref.items():
    plt.plot(hh["val_loss"], label=et, lw=1.2)
plt.xlabel("Época"); plt.ylabel("Pérdida de validación (pliegue 1)")
plt.title(f"Refinamiento de {MEJOR_DL}")
plt.legend(fontsize=7, ncol=2)
guardar_fig("f08b_refinamiento_dl")

## 9. Balanceo de clases
Primero se verifica el equilibrio de clases (sección 4). Aunque estén balanceadas, se evalúan **cuatro técnicas** para demostrarlo con datos: pesos de clase, sobremuestreo aleatorio (duplica registros de la clase minoritaria), submuestreo aleatorio (elimina registros de la mayoritaria) y SMOTE-NC (genera registros sintéticos para variables mixtas; Chawla et al., 2002). Cada técnica se aplica **solo a la parte de entrenamiento de cada pliegue** y se evalúa con la misma validación cruzada, para el mejor modelo de ML y el mejor de DL.

Regla de decisión: se adopta una técnica solo si mejora el F1 medio de "Sin balanceo" en más de una desviación estándar; si no, se mantiene sin balanceo.

In [ ]:
# ============================================================
# 9. BALANCEO DE CLASES
# ============================================================
ESTRATEGIAS = ["Sin balanceo", "Pesos de clase", "Sobremuestreo aleatorio", "Submuestreo aleatorio", "SMOTE-NC"]
filas, por_pliegue = [], []
for fam, nombre, fab in [("ML", MEJOR_ML, lambda: fabrica_ml(*mejores_ml[MEJOR_ML])),
                         ("DL", MEJOR_DL, lambda: fabrica_dl(MEJOR_DL, **CONFIG_DL))]:
    for est in ESTRATEGIAS:
        df_cv, _ = validar_cruzado(*fab(), estrategia=est)
        por_pliegue.append(df_cv.assign(Familia=fam, Modelo=nombre, Estrategia=est))
        filas.append({"Familia": fam, "Modelo": nombre, "Estrategia": est, **resumir_cv(df_cv)})
        print(f"{fam} · {est}: F1 CV = {filas[-1]['F1 media']:.4f}")
resumen_bal = pd.DataFrame(filas)
bal = pd.concat(por_pliegue)
guardar_tabla(bal.round(4), "t18_balanceo_por_pliegue")
guardar_tabla(resumen_bal.round(4), "t19_balanceo_resumen")
display(resumen_bal.round(4))

plt.figure(figsize=(11, 4.2))
sns.barplot(data=bal, x="Estrategia", y="F1", hue="Familia", errorbar="sd", palette=["#2a7fb8", "#e08a2c"], capsize=0.08)
plt.ylim(0, 1); plt.ylabel("F1 (media ± DE, 5 pliegues)"); plt.xlabel("")
plt.title(f"Efecto del balanceo · ML: {MEJOR_ML} · DL: {MEJOR_DL}")
guardar_fig("f10_balanceo")

ESTRATEGIA_FINAL = {}
for fam in ["ML", "DL"]:
    r = resumen_bal[resumen_bal.Familia == fam].set_index("Estrategia")
    base_m, base_sd = r.loc["Sin balanceo", "F1 media"], r.loc["Sin balanceo", "F1 DE"]
    mejor = r["F1 media"].idxmax()
    ESTRATEGIA_FINAL[fam] = mejor if r.loc[mejor, "F1 media"] - base_m > base_sd else "Sin balanceo"
print("Estrategia adoptada:", ESTRATEGIA_FINAL)
RESUMEN["balanceo_resumen"] = resumen_bal.round(4).to_dict(orient="records")
RESUMEN["estrategia_final"] = ESTRATEGIA_FINAL

## 10. Evaluación final en el 30 % de prueba: tabla de todos los resultados
Cada modelo se entrena una vez con **todo el 70 %** (con sus hiperparámetros elegidos por validación cruzada) y se evalúa **una sola vez** en el 30 % de prueba. La tabla reúne, para todos los modelos, las métricas de validación cruzada, de entrenamiento y de prueba. El **ganador** es el de mayor F1 en validación cruzada (criterio fijado antes de mirar la prueba). Para verificar que la máquina aprendió se compara con la línea base y se revisa la diferencia entre entrenamiento y prueba (sobreajuste).

In [ ]:
# ============================================================
# 10. EVALUACIÓN FINAL: TODOS LOS MODELOS EN EL 30 % DE PRUEBA
# ============================================================
finales = {}
# Modelos de ML
for nombre, (Clase, params) in mejores_ml.items():
    est = ESTRATEGIA_FINAL["ML"] if nombre == MEJOR_ML else "Sin balanceo"
    Xr, yr, cw = remuestrear(est, X_train, y_train)
    ajustar, predecir = fabrica_ml(Clase, params)
    obj = ajustar(Xr, yr, None, None, cw, SEMILLA)
    finales[nombre] = {"Familia": "ML", "obj": obj, "pred": predecir, "Balanceo": est,
                       "cv": tabla_ml.set_index("Modelo").loc[nombre]}
# Modelos de DL
for arq in ARQUITECTURAS:
    refinado = arq == MEJOR_DL
    est = ESTRATEGIA_FINAL["DL"] if refinado else "Sin balanceo"
    hp = CONFIG_DL if refinado else {}
    ep = EPOCAS_DL_REF if refinado else epocas_dl[arq]
    Xr, yr, cw = remuestrear(est, X_train, y_train)
    obj = entrenar_dl(arq, Xr, yr, cw=cw, epocas=ep, **hp)
    nombre = arq + (" (refinada)" if refinado else "")
    cv_row = (tabla_ref.iloc[0] if refinado else tabla_dl.set_index("Modelo").loc[arq])
    finales[nombre] = {"Familia": "DL", "obj": obj, "pred": predecir_dl, "Balanceo": est, "cv": cv_row, "Épocas": ep}

filas, preds_test = [], {}
for nombre, f in finales.items():
    p_tr, s_tr = f["pred"](f["obj"], X_train)
    p_te, s_te = f["pred"](f["obj"], X_test)
    preds_test[nombre] = (p_te, s_te)
    mtr, mte = metricas(y_train, p_tr, s_tr), metricas(y_test, p_te, s_te)
    cm = confusion_matrix(y_test, p_te)
    filas.append({"Familia": f["Familia"], "Modelo": nombre, "Balanceo": f["Balanceo"],
                  "F1 CV (media)": f["cv"]["F1 media"], "F1 CV (DE)": f["cv"]["F1 DE"],
                  "F1 entrenamiento": mtr["F1"],
                  **{f"{k} prueba": v for k, v in mte.items()},
                  "Brecha F1 (entren. − prueba)": mtr["F1"] - mte["F1"],
                  "VN": cm[0, 0], "FP": cm[0, 1], "FN": cm[1, 0], "VP": cm[1, 1]})
tabla_final = pd.DataFrame(filas)
no_base = ~tabla_final["Modelo"].str.startswith("Línea")
tabla_final["Ranking (F1 CV)"] = tabla_final["F1 CV (media)"].where(no_base).rank(ascending=False, method="min")
GANADOR = tabla_final.loc[tabla_final["Ranking (F1 CV)"] == 1, "Modelo"].iloc[0]
MEJOR_EN_PRUEBA = tabla_final[no_base].sort_values("F1 prueba", ascending=False).iloc[0]["Modelo"]
tabla_final["Resultado"] = np.where(tabla_final["Modelo"] == GANADOR, "GANADOR", "")
tabla_final = tabla_final.sort_values("Ranking (F1 CV)", na_position="last")
guardar_tabla(tabla_final.round(4), "t20_resultados_finales")
display(tabla_final.round(4))
print(f"\nGanador (criterio: F1 en validación cruzada): {GANADOR}")
print(f"Mejor F1 en prueba: {MEJOR_EN_PRUEBA}" + (" (coincide)" if MEJOR_EN_PRUEBA == GANADOR else " (no coincide; se comenta en el informe)"))
base = tabla_final[tabla_final["Modelo"].str.startswith("Línea")].iloc[0]
g = tabla_final[tabla_final["Modelo"] == GANADOR].iloc[0]
print(f"¿Aprendió? Exactitud ganador {g['Exactitud prueba']:.4f} vs línea base {base['Exactitud prueba']:.4f}")
RESUMEN["resultados_finales"] = tabla_final.round(4).to_dict(orient="records")
RESUMEN["ganador"] = GANADOR; RESUMEN["mejor_en_prueba"] = MEJOR_EN_PRUEBA

# Gráfico resumen de prueba
d = tabla_final[no_base].melt(id_vars="Modelo", value_vars=["Exactitud prueba", "F1 prueba", "ROC-AUC prueba"],
                              var_name="Métrica", value_name="Valor")
d["Métrica"] = d["Métrica"].str.replace(" prueba", "")
plt.figure(figsize=(11, 4.2))
sns.barplot(data=d, x="Modelo", y="Valor", hue="Métrica", palette=["#2a7fb8", "#e08a2c", "#4caf50"])
plt.axhline(base["Exactitud prueba"], ls="--", color="grey", lw=1, label="Exactitud línea base")
plt.xticks(rotation=20, ha="right"); plt.ylim(0, 1); plt.xlabel(""); plt.ylabel("Valor en prueba (30 %)")
plt.title(f"Resultados en prueba · ganador: {GANADOR}"); plt.legend(fontsize=8)
guardar_fig("f11_resultados_prueba")

# Matriz de confusión y ROC: ganador y mejor modelo de la otra familia
fam_g = finales[GANADOR]["Familia"]
otro = tabla_final[(tabla_final.Familia != fam_g) & no_base].iloc[0]["Modelo"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, nombre in zip(axes[:2], [GANADOR, otro]):
    sns.heatmap(confusion_matrix(y_test, preds_test[nombre][0]), annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Malo", "Bueno"], yticklabels=["Malo", "Bueno"])
    ax.set_title(f"Matriz de confusión · {nombre}"); ax.set_xlabel("Predicción"); ax.set_ylabel("Real")
for nombre in [GANADOR, otro]:
    fpr, tpr, _ = roc_curve(y_test, preds_test[nombre][1])
    axes[2].plot(fpr, tpr, label=f"{nombre} (AUC={roc_auc_score(y_test, preds_test[nombre][1]):.3f})")
axes[2].plot([0, 1], [0, 1], "--", color="grey", lw=0.8, label="Azar")
axes[2].set_xlabel("Tasa de falsos positivos"); axes[2].set_ylabel("Tasa de verdaderos positivos")
axes[2].set_title("Curvas ROC en prueba"); axes[2].legend(fontsize=8)
guardar_fig("f12_ganador_prueba")
print(classification_report(y_test, preds_test[GANADOR][0], target_names=["Malo", "Bueno"], digits=4))

## 11. Importancia de variables por permutación
Importancia por permutación del modelo ganador sobre el 30 % de prueba (interpretación posterior a la selección; no se usa para elegir modelos), agregada por variable original.

In [ ]:
# ============================================================
# 11. IMPORTANCIA DE VARIABLES Y EXPORTACIÓN
# ============================================================
import joblib
fg = finales[GANADOR]
rng = np.random.default_rng(SEMILLA)
f1_base = f1_score(y_test, fg["pred"](fg["obj"], X_test)[0])
filas = []
for c in X_test.columns:
    caidas = []
    for _ in range(20):
        Xp = X_test.copy(); Xp[c] = rng.permutation(Xp[c].values)
        caidas.append(f1_base - f1_score(y_test, fg["pred"](fg["obj"], Xp)[0]))
    filas.append({"Variable": c, "Importancia media (ΔF1)": np.mean(caidas), "DE": np.std(caidas)})
tabla_imp = pd.DataFrame(filas).sort_values("Importancia media (ΔF1)", ascending=False)
guardar_tabla(tabla_imp.round(4), "t21_importancia")
display(tabla_imp.round(4))
plt.figure(figsize=(7, 3.8))
plt.barh(tabla_imp["Variable"][::-1], tabla_imp["Importancia media (ΔF1)"][::-1], xerr=tabla_imp["DE"][::-1], color="#2a7fb8")
plt.xlabel("Caída media de F1 al permutar"); plt.title(f"Importancia por permutación · {GANADOR}")
guardar_fig("f13_importancia")
RESUMEN["importancia"] = tabla_imp.round(4).to_dict(orient="records")

## 12. Explicabilidad con SHAP
Se explica con SHAP (Lundberg y Lee, 2017) el **mejor modelo de ML**, que es el que se despliega en la aplicación (más liviano y explicable que las redes). Se calculan valores SHAP sobre el 30 % de prueba (datos no vistos): resumen global, importancia agregada por variable original (sumando las columnas one-hot de cada variable) y **casos particulares**: un héroe bien clasificado, un villano bien clasificado y un error.

In [ ]:
# ============================================================
# 12. EXPLICABILIDAD CON SHAP (modelo desplegado = mejor ML)
# ============================================================
prep_app, mod_app = finales[MEJOR_ML]["obj"]
nombres = list(prep_app.get_feature_names_out())
nombres_cortos = [n.split("__", 1)[-1] for n in nombres]

def variable_original(nombre_col):
    for v in CAT + NUM:
        if nombre_col == v or nombre_col.startswith(v + "_") or nombre_col == f"missingindicator_{v}":
            return v
    return nombre_col

Xte_app = pd.DataFrame(prep_app.transform(X_test), columns=nombres_cortos)
Xtr_app = pd.DataFrame(prep_app.transform(X_train), columns=nombres_cortos)
muestra = Xte_app.sample(n=min(300, len(Xte_app)), random_state=SEMILLA)

if isinstance(mod_app, (RandomForestClassifier, DecisionTreeClassifier)):
    explicador = shap.TreeExplainer(mod_app)
    sv = explicador(muestra)
    if sv.values.ndim == 3:                       # (n, variables, clases) -> clase "Bueno"
        sv = sv[:, :, 1]
else:
    f = (lambda Z: mod_app.predict_proba(Z)[:, 1]) if hasattr(mod_app, "predict_proba") else mod_app.decision_function
    fondo = shap.sample(Xtr_app, 100, random_state=SEMILLA)
    explicador = shap.Explainer(f, fondo, algorithm="permutation")
    sv = explicador(muestra)

plt.figure()
shap.plots.beeswarm(sv, max_display=15, show=False)
plt.title(f"SHAP: impacto de cada atributo en la probabilidad de «bueno» · {MEJOR_ML}")
guardar_fig("f14_shap_beeswarm")

# Importancia SHAP agregada por variable original
abs_media = pd.Series(np.abs(sv.values).mean(axis=0), index=nombres_cortos)
agregado = abs_media.groupby([variable_original(n) for n in nombres_cortos]).sum().sort_values(ascending=False)
tabla_shap = agregado.rename("|SHAP| medio").rename_axis("Variable").reset_index()
guardar_tabla(tabla_shap.round(4), "t22_shap_por_variable")
display(tabla_shap.round(4))
plt.figure(figsize=(7, 3.8))
plt.barh(tabla_shap["Variable"][::-1], tabla_shap["|SHAP| medio"][::-1], color="#2a7fb8")
plt.xlabel("|SHAP| medio (suma de sus columnas one-hot)"); plt.title("Importancia SHAP por variable original")
guardar_fig("f15_shap_por_variable")
RESUMEN["shap_por_variable"] = tabla_shap.round(4).to_dict(orient="records")

# Casos particulares (sobre la muestra explicada)
pred_m = mod_app.predict(muestra.values)
y_m = y_test[muestra.index.values]           # muestra conserva la posición en el conjunto de prueba
casos = {}
for etiqueta, cond in [("Héroe bien clasificado", (y_m == 1) & (pred_m == 1)),
                       ("Villano bien clasificado", (y_m == 0) & (pred_m == 0)),
                       ("Error: villano clasificado como héroe", (y_m == 0) & (pred_m == 1))]:
    idx = np.where(cond)[0]
    if len(idx):
        casos[etiqueta] = int(idx[0])
filas = []
for k, (etiqueta, i) in enumerate(casos.items()):
    fila_orig = X_test.iloc[muestra.index.values[i]]
    plt.figure()
    shap.plots.waterfall(sv[i], max_display=10, show=False)
    plt.title(etiqueta)
    guardar_fig(f"f16_shap_caso_{k+1}")
    contrib = pd.Series(sv.values[i], index=nombres_cortos).groupby([variable_original(n) for n in nombres_cortos]).sum()
    filas.append({"Caso": etiqueta, **{c: fila_orig[c] for c in CAT + NUM},
                  "Real": etq[int(y_m[i])], "Predicho": etq[int(pred_m[i])],
                  "Mayor aporte": contrib.abs().idxmax(), "Aporte (SHAP)": round(float(contrib[contrib.abs().idxmax()]), 4)})
tabla_casos = pd.DataFrame(filas)
guardar_tabla(tabla_casos, "t23_shap_casos")
display(tabla_casos)
RESUMEN["shap_casos"] = tabla_casos.astype(str).to_dict(orient="records")

## 13. Exportación de resultados y del modelo para la aplicación
Se guarda el modelo desplegado con su preprocesador, las opciones del formulario, las estadísticas de referencia del entrenamiento (para el monitoreo de deriva) y sus métricas en prueba (para el panel de monitoreo).

In [ ]:
# ============================================================
# 13. EXPORTACIÓN
# ============================================================
import joblib
opciones = {c: sorted(X_train[c].dropna().unique().tolist()) for c in CAT}
RESUMEN["opciones_formulario"] = opciones
fila_app = tabla_final[tabla_final["Modelo"] == MEJOR_ML].iloc[0]
referencia = {
    "faltantes_pct": {c: round(float(100 * X_train[c].isna().mean()), 2) for c in CAT + NUM},
    "mediana_log_apariciones": float(np.log1p(X_train["APPEARANCES"]).median()),
    "mediana_anio": float(X_train["YEAR"].median()),
    "prop_buenos_predicha_prueba": float(np.mean(preds_test[MEJOR_ML][0])),
    "moda": {c: X_train[c].mode().iloc[0] for c in CAT},
}
paquete = {
    "nombre_modelo": MEJOR_ML, "prep": prep_app, "modelo": mod_app,
    "variables": CAT + NUM, "categoricas": CAT, "numericas": NUM, "opciones": opciones,
    "referencia": referencia,
    "metricas_prueba": {k: float(fila_app[f"{k} prueba"]) for k in METRICAS},
    "fondo_shap": Xtr_app.sample(100, random_state=SEMILLA).values,
    "nombres_columnas": nombres_cortos,
    "versiones": {"scikit_learn": sklearn.__version__, "numpy": np.__version__},
}
joblib.dump(paquete, f"{OUT}/app_modelo.joblib")
print("Modelo para la aplicación:", MEJOR_ML, "| scikit-learn", sklearn.__version__)


with open(f"{OUT}/resumen.json", "w", encoding="utf-8") as f:
    json.dump(RESUMEN, f, ensure_ascii=False, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
with zipfile.ZipFile("salidas_acif104.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for raiz, _, archivos in os.walk(OUT):
        for a in archivos:
            z.write(os.path.join(raiz, a))
print("Exportado: salidas_acif104.zip")
try:
    from google.colab import files
    files.download("salidas_acif104.zip")
except Exception:
    pass